In [ ]:
%pip install scikit-learn

In [ ]:
### Desenha o mapa 2D do distrito do Porto com base em coordenadas geográficas (latitude/longitude) de imóveis.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Ler o CSV com separador padrão ',' (ignora linhas corrompidas se existirem)
df_raw = pd.read_csv('../data/porto_district_all_geocoded.csv', sep=',', on_bad_lines='skip')

# Mostrar as primeiras 5 linhas das colunas relevantes para confirmar a leitura
print(df_raw[['location', 'latitude', 'longitude']].head())

# 2. Converter para número sem restringir a [-90, 90] (caso estejam em metros/outra escala)
df = df_raw.copy()
for col in ['latitude', 'longitude']:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.'), errors='coerce'
    )

# Remover apenas valores nulos (NaN) ou zeros exatos
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]

print(f'\nTotal de pontos válidos: {len(df)}')

if df.empty:
    print(
        'AVISO: As colunas latitude/longitude estão vazias ou sem números'
        ' válidos neste CSV.'
    )
else:
    print(df[['latitude', 'longitude']].describe())

    # 3. Calcular o elemento central (mediana)
    lat_centro = df['latitude'].median()
    lon_centro = df['longitude'].median()

    # 4. Distância máxima ao centro para garantir que TODOS os pontos aparecem
    margem = 1.05
    max_dx = max((df['longitude'] - lon_centro).abs().max() * margem, 1e-5)
    max_dy = max((df['latitude'] - lat_centro).abs().max() * margem, 1e-5)

    # 5. Criar o mapa 2D
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(
        df['longitude'],
        df['latitude'],
        s=10,
        color='royalblue',
        alpha=0.5,
        label='Imóveis',
    )

    # 6. Centrar o enquadramento no elemento central
    ax.set_xlim(lon_centro - max_dx, lon_centro + max_dx)
    ax.set_ylim(lat_centro - max_dy, lat_centro + max_dy)

    # Ajusta proporção geográfica se estiver em graus; caso contrário usa proporção 1:1
    if -90 <= lat_centro <= 90:
        ax.set_aspect(1 / np.cos(np.radians(lat_centro)), adjustable='box')
    else:
        ax.set_aspect('equal', adjustable='box')

    ax.set_title('Mapa 2D — Distrito do Porto (Centrado)')
    ax.set_xlabel('Longitude (X)')
    ax.set_ylabel('Latitude (Y)')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
### Desenha o mapa 2D do distrito do Porto com base em coordenadas geográficas (latitude/longitude) de imóveis com o preço por metro quadrado.


import matplotlib.pyplot as plt
import numpy as np

# 1. Filtrar áreas > 0 (evitar divisão por zero) e calcular o preço por m²
df_plot = df_clean[df_clean["area_m2"] > 0].copy()
df_plot["price_m2"] = df_plot["price_eur"] / df_plot["area_m2"]

# Ordenar de forma ascendente para destacar as zonas mais caras por cima
df_plot = df_plot.sort_values("price_m2")

# 2. Definir limites de cor (percentis 2% e 98%) para evitar que outliers esmaguem a escala
vmin, vmax = df_plot["price_m2"].quantile([0.02, 0.98])

# 3. Calcular centro e enquadramento geográfico
lat_centro = df_plot["latitude"].median()
lon_centro = df_plot["longitude"].median()
margem = 1.05
max_dx = max((df_plot["longitude"] - lon_centro).abs().max() * margem, 1e-5)
max_dy = max((df_plot["latitude"] - lat_centro).abs().max() * margem, 1e-5)

# 4. Criar o mapa de dispersão com escala de cor
fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(
    df_plot["longitude"],
    df_plot["latitude"],
    c=df_plot["price_m2"],
    cmap="turbo",       # Alternativas: "viridis", "plasma", "RdYlBu_r"
    vmin=vmin,
    vmax=vmax,
    s=12,
    alpha=0.75,
)

# Legenda da escala de cor
cbar = fig.colorbar(sc, ax=ax, extend="both", shrink=0.85)
cbar.set_label("Preço por m² (€/m²)", fontsize=11)

# Enquadramento e proporção real do distrito
ax.set_xlim(lon_centro - max_dx, lon_centro + max_dx)
ax.set_ylim(lat_centro - max_dy, lat_centro + max_dy)
ax.set_aspect(1 / np.cos(np.radians(lat_centro)), adjustable="box")

ax.set_title("Distribuição Espacial do Preço por m² (€/m²) — Distrito do Porto")
ax.set_xlabel("Longitude (X)")
ax.set_ylabel("Latitude (Y)")
ax.grid(True, linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()